# Echo — Groupe 6

Faut-il généraliser les notifications agressives chez Echo ?

## Inspection des données

Les contrôles ci-dessous portent sur les CSV bruts. Aucune correction n'est appliquée.

In [1]:
from pathlib import Path
import pandas as pd

# Le notebook peut être ouvert depuis la racine du dépôt.
data_dir = Path("data")
if not data_dir.is_dir():
    raise FileNotFoundError("Ouvrir le notebook depuis le dossier GrowthModel.")

users_raw = pd.read_csv(data_dir / "users.csv", dtype={"user_id": "string"})
activity_raw = pd.read_csv(data_dir / "weekly_activity.csv", dtype={"user_id": "string"})
spend_raw = pd.read_csv(data_dir / "marketing_spend.csv")
tables = {"users": users_raw, "weekly_activity": activity_raw, "marketing_spend": spend_raw}

for name, table in tables.items():
    print(f"{name} : {len(table):,} lignes, {len(table.columns)} colonnes")
    print(table.dtypes.to_string())
    print()


users : 20,909 lignes, 7 colonnes
user_id                 string[python]
signup_date                     object
acquisition_channel             object
platform                        object
age_bucket                      object
push_variant                    object
shows_followed_week1             int64

weekly_activity : 334,399 lignes, 6 colonnes
user_id                        string[python]
week_start                             object
sessions                                int64
listening_minutes                     float64
ad_impressions                          int64
push_notifications_received             int64

marketing_spend : 17 lignes, 3 colonnes
month         object
channel       object
spend_eur    float64



### Valeurs manquantes et doublons

In [2]:
keys = {"users": ["user_id"], "weekly_activity": ["user_id", "week_start"], "marketing_spend": ["month", "channel"]}
audit_rows = []
for name, table in tables.items():
    distinct_rows = table.drop_duplicates()
    audit_rows.append({
        "table": name,
        "lignes": len(table),
        "copies_exactes": int(table.duplicated().sum()),
        "cles_repetees": int(table.duplicated(keys[name]).sum()),
        "lignes_en_conflit_apres_dedoublonnage": int(distinct_rows.duplicated(keys[name], keep=False).sum()),
    })
    print(f"Valeurs manquantes — {name}")
    print(table.isna().sum().to_string())
    print()
print(pd.DataFrame(audit_rows).to_string(index=False))


Valeurs manquantes — users
user_id                   0
signup_date               0
acquisition_channel       0
platform                  0
age_bucket              660
push_variant              0
shows_followed_week1      0

Valeurs manquantes — weekly_activity
user_id                        0
week_start                     0
sessions                       0
listening_minutes              0
ad_impressions                 0
push_notifications_received    0

Valeurs manquantes — marketing_spend
month        0
channel      0
spend_eur    0

          table  lignes  copies_exactes  cles_repetees  lignes_en_conflit_apres_dedoublonnage
          users   20909             124            124                                      0
weekly_activity  334399            1663           1663                                      0
marketing_spend      17               0              0                                      0


### Dates et catégories

Les dates avec `/` sont lues au format jour/mois/année. Les jours supérieurs à 12 permettent de vérifier cette convention. Les colonnes de contrôle ne remplacent pas les données brutes.

In [3]:
slash_dates = users_raw["signup_date"].str.contains("/", regex=False)
signup_dates = pd.to_datetime(users_raw["signup_date"], format="%Y-%m-%d", errors="coerce")
signup_dates.loc[slash_dates] = pd.to_datetime(users_raw.loc[slash_dates, "signup_date"], format="%d/%m/%Y", errors="coerce")
week_dates = pd.to_datetime(activity_raw["week_start"], format="%Y-%m-%d", errors="coerce")
month_dates = pd.to_datetime(spend_raw["month"], format="%Y-%m-%d", errors="coerce")

print("Dates d'inscription avec / :", int(slash_dates.sum()))
print("Dont jour supérieur à 12 :", int((users_raw.loc[slash_dates, "signup_date"].str[:2].astype(int) > 12).sum()))
print("Dates invalides (inscription, semaine, dépense) :", int(signup_dates.isna().sum()), int(week_dates.isna().sum()), int(month_dates.isna().sum()))
print("Inscriptions :", signup_dates.min().date(), "à", signup_dates.max().date())
print("Activité :", week_dates.min().date(), "à", week_dates.max().date())
print("Semaines ne commençant pas un lundi :", int((week_dates.dt.dayofweek != 0).sum()))
print("Dépenses non datées du premier du mois :", int((month_dates.dt.day != 1).sum()))
for column in ["acquisition_channel", "platform", "age_bucket", "push_variant"]:
    print(f"\n{column}")
    print(users_raw[column].value_counts(dropna=False).to_string())


Dates d'inscription avec / : 314
Dont jour supérieur à 12 : 193
Dates invalides (inscription, semaine, dépense) : 0 0 0
Inscriptions : 2025-01-01 à 2026-06-30
Activité : 2024-12-30 à 2026-06-29
Semaines ne commençant pas un lundi : 0
Dépenses non datées du premier du mois : 0

acquisition_channel
acquisition_channel
paid_social            8950
organic_store          7722
podcast_cross_promo    4237

platform
platform
Android    10496
iOS        10413

age_bucket
age_bucket
25-34    7043
18-24    5964
35-44    4143
45+      3099
NaN       660

push_variant
push_variant
standard           18181
control             1276
aggressive_push     1240
Control              100
AGGRESSIVE_PUSH       58
aggressive            54


### Valeurs numériques

In [4]:
print(activity_raw.select_dtypes("number").describe(percentiles=[0.5, 0.99]).round(2).to_string())
print("\nDurées négatives :", int((activity_raw["listening_minutes"] < 0).sum()))
# Une semaine entière représente 10 080 minutes.
print("Durées supérieures à une semaine :", int((activity_raw["listening_minutes"] > 7 * 24 * 60).sum()))
print("Émissions suivies en semaine 1 :")
print(users_raw["shows_followed_week1"].describe().to_string())
for name, table, columns in [
    ("users", users_raw, ["shows_followed_week1"]),
    ("weekly_activity", activity_raw, ["sessions", "ad_impressions", "push_notifications_received"]),
    ("marketing_spend", spend_raw, ["spend_eur"]),
]:
    print(name, "— valeurs négatives :", (table[columns] < 0).sum().to_dict())


        sessions  listening_minutes  ad_impressions  push_notifications_received
count  334399.00          334399.00       334399.00                    334399.00
mean        3.51             114.53           11.56                         5.26
std         1.84            1737.45            6.88                         6.23
min         1.00              -1.00            1.00                         0.00
50%         3.00              70.10           10.00                         4.00
99%         9.00             264.30           32.00                        36.00
max        21.00           99999.00           82.00                        58.00

Durées négatives : 96
Durées supérieures à une semaine : 100
Émissions suivies en semaine 1 :
count    20909.000000
mean         2.211010
std          1.551813
min          0.000000
25%          1.000000
50%          2.000000
75%          3.000000
max         12.000000
users — valeurs négatives : {'shows_followed_week1': 0}
weekly_activity — valeurs

### Cohérence entre les fichiers et couverture

Une semaine sans ligne signifie qu'aucune activité n'est enregistrée. Elle ne prouve pas un départ définitif. La première et la dernière semaine peuvent être partielles.

In [5]:
print("Utilisateurs de l'activité absents de users :", len(set(activity_raw["user_id"]) - set(users_raw["user_id"])))
print("Utilisateurs sans activité :", len(set(users_raw["user_id"]) - set(activity_raw["user_id"])))

# Dédupliquer seulement la table de correspondance pour éviter une jointure plusieurs-à-plusieurs.
signup_lookup = pd.DataFrame({"user_id": users_raw["user_id"], "signup_date": signup_dates}).drop_duplicates()
joined = activity_raw[["user_id", "week_start"]].merge(signup_lookup, on="user_id", how="left", validate="many_to_one")
signup_monday = joined["signup_date"] - pd.to_timedelta(joined["signup_date"].dt.dayofweek, unit="D")
print("Activités antérieures à la semaine d'inscription :", int((pd.to_datetime(joined["week_start"]) < signup_monday).sum()))
print("Activités après l'extraction du 30/06/2026 :", int((week_dates > pd.Timestamp("2026-06-30")).sum()))

all_months = pd.date_range("2025-01-01", "2026-06-01", freq="MS")
for channel in spend_raw["channel"].unique():
    recorded_months = month_dates[spend_raw["channel"] == channel]
    print("Mois de dépenses absents pour", channel, ":", all_months.difference(recorded_months).strftime("%Y-%m").tolist())
print("Canaux sans dépenses renseignées :", sorted(set(users_raw["acquisition_channel"]) - set(spend_raw["channel"])))

test_users = pd.DataFrame({"signup_date": signup_dates, "push_variant": users_raw["push_variant"]}).loc[users_raw["push_variant"] != "standard"]
outside_test = ~test_users["signup_date"].between("2026-01-15", "2026-03-15")
print("Lignes de participants inscrits hors de la période du test :", int(outside_test.sum()))


Utilisateurs de l'activité absents de users : 0
Utilisateurs sans activité : 0
Activités antérieures à la semaine d'inscription : 0
Activités après l'extraction du 30/06/2026 : 0
Mois de dépenses absents pour paid_social : ['2025-05']
Canaux sans dépenses renseignées : ['organic_store', 'podcast_cross_promo']
Lignes de participants inscrits hors de la période du test : 0


### Bilan

- 124 copies exactes dans `users` et 1 663 dans `weekly_activity`, sans conflit de valeurs sur les clés.
- 314 dates d'inscription au format jour/mois/année.
- Variantes à harmoniser : `Control`, `AGGRESSIVE_PUSH` et `aggressive`.
- 660 âges absents dans le brut.
- 96 durées négatives et 100 durées de 99 999 minutes dans le brut.
- Dépense `paid_social` de mai 2025 absente. Aucun coût renseigné pour les autres canaux.
- Aucun indicateur fiable ne permet d'identifier des comptes internes ou de test. Aucun compte n'est exclu sur une simple suspicion.

Les règles de nettoyage sont à valider avant leur application. Les effectifs changeront après retrait des copies exactes.